In [4]:
import degirum as dg
import numpy as np
import math 
import time
import cv2
from PIL import Image
from postprocess_utils import *

In [5]:
zoo=dg.connect_model_zoo("0.0.0.0:8778")
model = zoo.load_model("yolov8n-seg")
res = model("../resized_mask1.jpg")

In [6]:
img_src = cv2.imread("../resized_mask1.jpg") ## Resized image to 640X640
# print ("original image shape--------------",img_src.shape)
pred_order =[]

for el in res.results:
    prediction = el["data"]
    pred_order.append(prediction)
    

preds_decoded = decode_bbox(pred_order[:6], [1, 3, 640, 640])
mask = pred_order[6]
proto = pred_order[7]
proto = proto.transpose(0,3,1,2)
nc = preds_decoded.shape[1] - 4
preds_decoded = np.concatenate([preds_decoded, np.transpose(mask,(0, 2, 1))], 1)

p = non_max_suppression(preds_decoded,conf_thres=0.25, iou_thres=0.7, classes=None, agnostic=False, multi_label=False, max_det=300, nc=nc)

pred = np.vstack(p)
masks = process_mask(proto[0], pred[:, 6:], pred[:, :4],img_src.shape[:2], upsample=True)  # HWC
masks = np.moveaxis(masks, 0, -1) # masks, (H, W, N)
# rescale masks to original image
masks = scale_image(masks,img_src.shape)
masks = np.moveaxis(masks, -1, 0) # masks, (N, H, W))
image_with_masks = np.copy(img_src)
for mask_i in masks:
    image_with_masks = overlay(image_with_masks, mask_i, color=(255, 0, 0), alpha=0.4)

cv2.imwrite('mask_pred.PNG', image_with_masks)


## Drawing the boxes

# pred[:, :4] = scale_boxes(img_src.shape[:2],pred[:, :4] , img_src.shape)
# for box in pred[:, :4]:
#     p1, p2 = (int(box[0]), int(box[1])), (int(box[2]), int(box[3]))
#     cv2.rectangle(img_src, p1, p2, (0,255,0), thickness=2, lineType=cv2.LINE_AA)
# cv2.imwrite("box_pred.jpg", img_src)

original image shape-------------- (640, 640, 3)


True